# <span style="color:blue">PART 2: REST API INTEGRATION</span>


---
## 2.1 Understanding APIs: The Restaurant Analogy

Imagine a **restaurant**:

| Concept | API Equivalent |
|---|---|
| 🍳 Kitchen | Server with data |
| 📋 Menu | API documentation |
| 🧑‍🍳 Waiter | API endpoints |
| 📝 Your order | HTTP request |
| 🍽️ Your food | HTTP response |

> You **don't** go into the kitchen (database). You tell the waiter what you want, and they bring it to you.

---

### <span style="color:blue">HTTP Methods = Actions</span>

| Method | Meaning | Description |
|---|---|---|
| **`GET`** | "Show me the menu" | Retrieve data |
| **`POST`** | "I want to order this" | Create new data |
| **`PUT`** | "Change my entire order" | Replace data |
| **`PATCH`** | "Add extra cheese" | Update part of data |
| **`DELETE`** | "Cancel my order" | Remove data |

---

### <span style="color:blue">Status Codes = Kitchen's Response</span>

| Code | Message | Meaning |
|---|---|---|
| **200** | OK | "Here's your food!" |
| **201** | Created | "Order placed successfully!" |
| <span style="color:red">**400**</span> | Bad Request | "That's not on the menu" |
| <span style="color:red">**401**</span> | Unauthorized | "You need to pay first" |
| <span style="color:red">**404**</span> | Not Found | "We don't have that dish" |
| <span style="color:red">**429**</span> | Too Many Requests | "You're ordering too fast!" |
| <span style="color:red">**500**</span> | Server Error | "Kitchen is on fire" |

---
## 2.2 Real Example: GitHub API

**GitHub** provides free APIs to access public repository data. Let's explore!

> 📌 We use the `requests` library to make HTTP calls and `pandas` to work with the data.

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime


# Example 1: Get repository information
def get_repo_info(owner, repo):
    response = requests.get(f'https://api.github.com/repos/{owner}/{repo}')
    """
    Fetch information about a GitHub repository.

    Args:
        owner: Repository owner (e.g., 'pandas-dev')
        repo: Repository name (e.g., 'pandas')

    Returns:
        dict: Repository information
    """
    # API endpoint - GitHub's REST API base URL + path to specific repo
    url = f'https://api.github.com/repos/{owner}/{repo}'

    # Make GET request to the API endpoint
    response = requests.get(url)

    # Check status code to determine if request succeeded
    print(f"Status Code: {response.status_code}")
    print(f"Response Headers:")
    for key, value in list(response.headers.items())[:5]:
        print(f"  {key}: {value}")

    # Parse JSON response only if request was successful
    if response.status_code == 200:
        data = response.json()

        # Extract only the relevant fields we care about
        repo_info = {
            'name': data['name'],
            'full_name': data['full_name'],
            'description': data['description'],
            'stars': data['stargazers_count'],
            'forks': data['forks_count'],
            'watchers': data['watchers_count'],
            'open_issues': data['open_issues_count'],
            'language': data['language'],
            'created_at': data['created_at'],
            'updated_at': data['updated_at'],
            'size': data['size'],
            'license': data['license']['name'] if data['license'] else 'No license',
        }
        return repo_info
    else:
        print(f"Error: {response.status_code}")
        return None


# Try it!
repo_info = get_repo_info('pandas-dev', 'pandas')

if repo_info:
    print("\n=== Repository Information ===")
    for key, value in repo_info.items():
        print(f"{key}: {value}")

Status Code: 200
Response Headers:
  Date: Thu, 05 Mar 2026 19:30:37 GMT
  Content-Type: application/json; charset=utf-8
  Cache-Control: public, max-age=60, s-maxage=60
  Vary: Accept,Accept-Encoding, Accept, X-Requested-With
  ETag: W/"2e37bb0c0abd717f5daeebcac3c9bc3e1bd06e9efd3f928ec15ebf1e9dadb75d"

=== Repository Information ===
name: pandas
full_name: pandas-dev/pandas
description: Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more
stars: 48061
forks: 19721
watchers: 48061
open_issues: 3708
language: Python
created_at: 2010-08-24T01:37:33Z
updated_at: 2026-03-05T16:43:11Z
size: 384434
license: BSD 3-Clause "New" or "Revised" License


### Expected Output:
```
Status Code: 200
Response Headers:
  Content-Type: application/json
  X-RateLimit-Limit: 60
  X-RateLimit-Remaining: 59
  ...

=== Repository Information ===
name: pandas
full_name: pandas-dev/pandas
description: Flexible and powerful data analysis / manipulation library for Python
stars: 43256
forks: 17843
...
```

---

### <span style="color:blue">Understanding the Response</span>

An **HTTP Response** has 3 parts:

```
1. Status Line:   HTTP/1.1 200 OK

2. Headers:
   Content-Type: application/json
   X-RateLimit-Limit: 60        <-- Max requests per hour
   X-RateLimit-Remaining: 59    <-- Requests left

3. Body (JSON):
   {
     "name": "pandas",
     "stars": 43256,
     ...
   }
```

---
## 2.3 Handling Authentication

Many APIs require **authentication** to:
1. Track usage
2. Prevent abuse
3. Provide personalized data

---

### <span style="color:blue">Types of Authentication</span>

#### **Type 1: API Key in Header** *(Most Common)*

In [2]:
import os
from dotenv import load_dotenv  # pip install python-dotenv

# Load API key from .env file (keeps secrets out of your code)
load_dotenv()

api_key = os.getenv('GITHUB_TOKEN')

# Add the token to request headers for authentication
headers = {
    'Authorization': f'Bearer {api_key}',  # Token-based auth
    'Accept': 'application/vnd.github.v3+json',  # Tell server what format we want
    'User-Agent': 'Library-Tutorial-App',  # Identify your app
}

url = 'https://api.github.com/repos/pandas-dev/pandas'
response = requests.get(url, headers=headers)

#### Create a `.env` file in your project directory:

```
GITHUB_TOKEN=ghp_your_token_here
OPENWEATHER_KEY=your_key_here
```

#### <span style="color:red">⚠️ Why `.env`?</span>
- **Never** hardcode secrets in code!
- Different keys for dev/production
- Keeps secrets out of version control

#### Add to `.gitignore`:
```
.env
*.env
```

---

#### **Type 2: API Key in Query Parameters**

In [3]:
# Example: OpenWeather API uses key as a query parameter
api_key = os.getenv('OPENWEATHER_KEY')

# Parameters are automatically appended to the URL as ?q=Cairo&appid=...&units=metric
params = {
    'q': 'Cairo',
    'appid': api_key,
    'units': 'metric',  # Celsius instead of Kelvin
}

response = requests.get(
    'https://api.openweathermap.org/data/2.5/weather', params=params
)

#### **Type 3: OAuth** *(Complex but Secure)*

In [ ]:
# OAuth flow (simplified):
# 1. User authorizes your app
# 2. You get an access token
# 3. Use token for requests
# This is beyond our scope but good to know!

---
## <span style="color:blue">2.4 Advanced: Pagination</span>

**Problem**: API returns 100 results, but there are **10,000!**

**Solution**: **Pagination** — fetch data in multiple pages, just like a book.

### <span style="color:blue">Pagination Strategies</span>

**1. Page-based (GitHub):**
```
/repos?page=1&per_page=100
/repos?page=2&per_page=100
```

**2. Offset-based:**
```
/repos?offset=0&limit=100
/repos?offset=100&limit=100
```

**3. Cursor-based** *(most efficient)*:
```
/repos?cursor=abc123
/repos?cursor=def456

Response includes next cursor:
{
  "data": [...],
  "next_cursor": "def456"
}
```

In [4]:
import time


def get_all_repos(org_name, max_pages=None):
    """
    Fetch all repositories for an organization using pagination.
    GitHub API returns 30 repos per page by default.
    """
    all_repos = []
    page = 1

    while True:
        # Check if we've reached max_pages (useful for testing/limiting)
        if max_pages and page > max_pages:
            break

        print(f"Fetching page {page}...")

        # Add page parameter to tell the API which chunk of results we want
        params = {'page': page, 'per_page': 100}  # Max allowed by GitHub

        url = f'https://api.github.com/orgs/{org_name}/repos'
        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()

        # If no data returned, we've gone past the last page
        if not data or len(data) == 0:
            print("No more results!")
            break

        all_repos.extend(data)
        page += 1

        # Be polite - wait between requests to avoid rate limiting
        time.sleep(0.5)

    return all_repos


# Example: Get all pandas-dev repos (limit to 3 pages for demo)
repos = get_all_repos('pandas-dev', max_pages=3)
print(f"\nFetched {len(repos)} repositories")

# Convert list of dicts to a clean DataFrame
df = pd.DataFrame(
    [
        {
            'name': repo['name'],
            'stars': repo['stargazers_count'],
            'language': repo['language'],
            'description': repo['description'],
        }
        for repo in repos
    ]
)

print("\nTop 10 by Stars:")
print(df.sort_values('stars', ascending=False).head(10))

Fetching page 1...
Fetching page 2...
No more results!

Fetched 15 repositories

Top 10 by Stars:
                  name  stars          language  \
0               pandas  48061            Python   
9         pandas-stubs    310            Python   
1    pandas-governance     36              None   
2       pandas-msgpack     24            Python   
7  pandas-user-surveys     10  Jupyter Notebook   
8    pandas-dev-flaker     10            Python   
5          pandas-blog      8            Python   
3        pandas-compat      7            Python   
4       pandas-release      7            Python   
6              .github      3              None   

                                         description  
0  Flexible and powerful data analysis / manipula...  
9                       Public type stubs for pandas  
1  Project governance documents for the pandas Pr...  
2                                     Pandas Msgpack  
7                                               None  
8         

---
## 2.5 Rate Limiting & Retry Logic

<span style="color:red">**Problem**</span>: APIs limit requests to prevent abuse.

### GitHub Rate Limits:
- **Unauthenticated**: `60` requests/hour
- **Authenticated**: `5,000` requests/hour

---

### <span style="color:blue">Custom Rate Limiter Class</span>

In [5]:
import time
from datetime import datetime


class RateLimiter:
    """
    Smart rate limiter that tracks API usage.
    Uses a sliding time window to count recent requests.
    """

    def __init__(self, max_requests=60, time_window=3600):
        """
        Args:
            max_requests: Maximum requests allowed in the time window
            time_window: Time window in seconds (3600 = 1 hour)
        """
        self.max_requests = max_requests
        self.time_window = time_window
        self.requests = []  # List of timestamps of past requests

    def wait_if_needed(self):
        """Wait if we've hit the rate limit before making a new request."""
        now = time.time()

        # Remove old timestamps outside the sliding time window
        self.requests = [
            req_time for req_time in self.requests if now - req_time < self.time_window
        ]

        # If we've used up our quota, sleep until the oldest request expires
        if len(self.requests) >= self.max_requests:
            oldest_request = self.requests[0]
            sleep_time = self.time_window - (now - oldest_request)
            if sleep_time > 0:
                print(
                    f"⏰ Rate limit reached. Sleeping for {sleep_time:.1f} seconds..."
                )
                time.sleep(sleep_time)
            self.requests = []  # Clear after sleeping

        # Record the timestamp of this new request
        self.requests.append(now)


# Usage: 10 requests per minute limit
limiter = RateLimiter(max_requests=10, time_window=60)

url = 'https://api.github.com/repos/pandas-dev/pandas'
for i in range(15):
    limiter.wait_if_needed()  # Automatically pauses if limit is hit
    response = requests.get(url)
    print(f"Request {i+1} completed")

Request 1 completed
Request 2 completed
Request 3 completed
Request 4 completed
Request 5 completed
Request 6 completed
Request 7 completed
Request 8 completed
Request 9 completed
Request 10 completed
⏰ Rate limit reached. Sleeping for 56.9 seconds...
Request 11 completed
Request 12 completed
Request 13 completed
Request 14 completed
Request 15 completed


### <span style="color:blue">Checking API Limits from Headers</span>

In [6]:
def check_rate_limit(response):
    """
    Check rate limit info from response headers.
    GitHub includes rate limit details in every response.
    """
    if 'X-RateLimit-Limit' in response.headers:
        limit = int(response.headers['X-RateLimit-Limit'])
        remaining = int(response.headers['X-RateLimit-Remaining'])
        reset_timestamp = int(response.headers['X-RateLimit-Reset'])
        reset_time = datetime.fromtimestamp(reset_timestamp)

        print(f"Rate Limit: {remaining}/{limit}")
        print(f"Resets at: {reset_time}")

        # Warn when running low on available requests
        if remaining < 10:
            print("⚠️ Warning: Low on API requests!")

        return remaining
    return None


url = 'https://api.github.com/repos/pandas-dev/pandas'
response = requests.get(url)
check_rate_limit(response)

Rate Limit: 41/60
Resets at: 2026-03-05 22:30:36


41

### <span style="color:blue">Automatic Retry with Exponential Backoff</span>

**What is Exponential Backoff?**

Instead of hammering the server after a failure, we **wait progressively longer** between retries:

```
Attempt 1: Fails → Wait  1 second
Attempt 2: Fails → Wait  2 seconds
Attempt 3: Fails → Wait  4 seconds
Attempt 4: Fails → Wait  8 seconds
Attempt 5: Success! ✅
```

In [7]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def create_robust_session():
    """
    Create requests session with automatic retry logic.
    The session will automatically retry failed requests using exponential backoff.
    """
    session = requests.Session()

    # Define retry strategy - what to retry, how many times, and how long to wait
    retry_strategy = Retry(
        total=5,  # Maximum number of retries
        backoff_factor=1,  # Wait 0, 1, 2, 4, 8 seconds between retries
        status_forcelist=[429, 500, 502, 503, 504],  # Retry on these status codes
        allowed_methods=["HEAD", "GET", "OPTIONS", "POST"],  # Methods to retry
    )

    # Mount the retry adapter for both http and https
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)

    return session


# Usage - just use this session like you would requests.get()
session = create_robust_session()
response = session.get('https://api.github.com/repos/pandas-dev/pandas')
print(f"Status: {response.status_code}")

Status: 200


---
## 2.6 Error Handling & Logging

**Robust code** anticipates failures and records what happened for debugging.

In [8]:
import logging
from datetime import datetime

# Configure logging to write to both a file AND the console
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('api_requests.log'),  # Saves to disk
        logging.StreamHandler(),  # Also print to console
    ],
)

logger = logging.getLogger(__name__)


def fetch_with_error_handling(url, max_retries=3):
    """
    Robust API fetch with comprehensive error handling.
    Handles: timeouts, connection errors, rate limits, server errors, bad JSON.
    """
    for attempt in range(max_retries):
        try:
            logger.info(f"Attempt {attempt + 1}/{max_retries}: GET {url}")
            response = requests.get(url, timeout=10)  # Don't wait more than 10 seconds

            # Handle each status code type differently
            if response.status_code == 200:
                logger.info(f"✓ Success: {url}")
                return response.json()

            elif response.status_code == 404:
                logger.error(f"✗ Not Found: {url}")
                return None  # No point retrying - resource doesn't exist

            elif response.status_code == 429:
                # Server tells us how long to wait in the Retry-After header
                retry_after = int(response.headers.get('Retry-After', 60))
                logger.warning(f"Rate limited. Waiting {retry_after}s...")
                time.sleep(retry_after)
                continue

            elif response.status_code >= 500:
                # Server error - retry with exponential backoff
                logger.error(f"Server error ({response.status_code}). Retrying...")
                time.sleep(2**attempt)  # 1, 2, 4 seconds
                continue

            else:
                logger.error(f"HTTP {response.status_code}: {url}")
                response.raise_for_status()

        except requests.exceptions.Timeout:
            logger.warning(f"Timeout on attempt {attempt + 1}")
            if attempt < max_retries - 1:
                time.sleep(2**attempt)

        except requests.exceptions.ConnectionError as e:
            logger.error(f"Connection error: {e}")
            if attempt < max_retries - 1:
                time.sleep(2**attempt)

        except json.JSONDecodeError:
            logger.error("Invalid JSON response")  # Server returned non-JSON
            return None

        except Exception as e:
            logger.error(f"Unexpected error: {e}")
            return None

    logger.error(f"Failed after {max_retries} attempts")
    return None


# Usage
data = fetch_with_error_handling('https://api.github.com/repos/python/cpython')
if data:
    print(f"Repo: {data['full_name']}, Stars: {data['stargazers_count']}")

2026-03-05 22:16:07,996 - __main__ - INFO - Attempt 1/3: GET https://api.github.com/repos/python/cpython
--- Logging error ---
Traceback (most recent call last):
  File "c:\Users\PC\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "c:\Users\PC\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1256.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2713' in position 44: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\PC\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\PC\AppData\Roaming\Python\Python311\site-packages\traitlets\c

Repo: python/cpython, Stars: 71844


**Expected Log Output:**
```
2024-03-15 10:30:45 - __main__ - INFO - Attempt 1/3: GET https://api.github.com/...
2024-03-15 10:30:46 - __main__ - INFO - ✓ Success: https://api.github.com/...
```

---
## 2.7 Building a Reusable API Client

Combining everything we've learned into a **clean, professional `GitHubAPI` class** that can be reused across projects.

In [9]:
class GitHubAPI:
    """
    Reusable GitHub API client with all best practices:
    - Session management with retry logic
    - Rate limiting
    - Authentication via token
    - Logging
    """

    def __init__(self, token=None):
        self.base_url = 'https://api.github.com'
        self.session = self._create_session()  # Robust session with retries
        self.rate_limiter = RateLimiter(
            max_requests=5000, time_window=3600
        )  # Authenticated limits

        # Add authentication token if provided
        if token:
            self.session.headers.update({'Authorization': f'Bearer {token}'})

        # Always set these headers for proper API communication
        self.session.headers.update(
            {
                'Accept': 'application/vnd.github.v3+json',
                'User-Agent': 'Library-Tutorial/1.0',
            }
        )

        self.logger = logging.getLogger(self.__class__.__name__)

    def _create_session(self):
        """Create session with retry logic (private method)."""
        session = requests.Session()
        retry_strategy = Retry(
            total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504]
        )
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        return session

    def get(self, endpoint, params=None):
        """Make GET request with rate limiting."""
        self.rate_limiter.wait_if_needed()  # Respect rate limits
        url = f"{self.base_url}/{endpoint.lstrip('/')}"

        try:
            response = self.session.get(url, params=params, timeout=10)
            response.raise_for_status()  # Raises exception for 4xx/5xx
            self.logger.info(f"GET {endpoint} - Status: {response.status_code}")

            # Peek at remaining rate limit with each response
            remaining = check_rate_limit(response)
            return response.json()

        except Exception as e:
            self.logger.error(f"Error fetching {endpoint}: {e}")
            raise

    def get_repo(self, owner, repo):
        """Get repository information."""
        return self.get(f'/repos/{owner}/{repo}')

    def get_user_repos(self, username):
        """Get all repositories for a user."""
        return self.get(f'/users/{username}/repos', params={'per_page': 100})

    def search_repos(self, query, language=None, min_stars=None):
        """
        Search repositories.

        Args:
            query: Search query string
            language: Filter by programming language
            min_stars: Minimum stars required

        Returns:
            list: Repository results
        """
        # Build search query by combining filters with spaces
        q_parts = [query]
        if language:
            q_parts.append(f"language:{language}")
        if min_stars:
            q_parts.append(f"stars:>={min_stars}")

        q = ' '.join(q_parts)
        results = self.get('/search/repositories', params={'q': q})
        return results['items']

    def to_dataframe(self, repos):
        """Convert repository list to DataFrame for analysis."""
        data = []
        for repo in repos:
            data.append(
                {
                    'name': repo['name'],
                    'full_name': repo['full_name'],
                    'description': repo.get('description'),
                    'stars': repo['stargazers_count'],
                    'forks': repo['forks_count'],
                    'language': repo.get('language'),
                    'created_at': repo['created_at'],
                    'updated_at': repo['updated_at'],
                }
            )
        return pd.DataFrame(data)


# ============================
# Usage Examples
# ============================

api = GitHubAPI(token=os.getenv('GITHUB_TOKEN'))  # Authenticated (5000 req/hr)

# Get a single repository
repo = api.get_repo('pandas-dev', 'pandas')
print(f"Stars: {repo['stargazers_count']}")

# Search repositories with filters
python_repos = api.search_repos('machine learning', language='python', min_stars=1000)
df = api.to_dataframe(python_repos)
print(df.head())

2026-03-05 22:23:13,357 - GitHubAPI - INFO - GET /repos/pandas-dev/pandas - Status: 200


Rate Limit: 38/60
Resets at: 2026-03-05 22:30:36
Stars: 48061


2026-03-05 22:23:14,290 - GitHubAPI - INFO - GET /search/repositories - Status: 200


Rate Limit: 9/10
Resets at: 2026-03-05 22:24:14
⚠️ Warning: Low on API requests!
                        name                                 full_name  \
0   awesome-machine-learning     josephmisiti/awesome-machine-learning   
1            MachineLearning                      wepe/MachineLearning   
2           Machine-Learning             Jack-Cherish/Machine-Learning   
3     MachineLearning_Python          lawlite19/MachineLearning_Python   
4  machine_learning_examples  lazyprogrammer/machine_learning_examples   

                                         description  stars  forks language  \
0  A curated list of awesome Machine Learning fra...  71848  15332   Python   
1           Basic Machine Learning and Deep Learning   5646   3212   Python   
2  :zap:机器学习实战（Python3）：kNN、决策树、贝叶斯、逻辑回归、SVM、线性回归...  10239   5109   Python   
3                                     机器学习算法python实现   8363   2511   Python   
4  A collection of machine learning examples and ...   8828   6448   Python   


---
## 2.8 Working with Different Response Formats

APIs don't always respond with JSON — here's how to handle the two most common formats.

---

### <span style="color:blue">JSON (Most Common)</span>

In [10]:
# Simple JSON - direct parsing
response = requests.get('https://api.github.com/repos/pandas-dev/pandas')
data = response.json()

# Nested JSON - requires flattening for DataFrame use
data = {
    "user": {
        "name": "Ahmed",
        "address": {"city": "Cairo", "country": "Egypt"},
        "repositories": [
            {"name": "repo1", "stars": 10},
            {"name": "repo2", "stars": 25},
        ],
    }
}

# Flatten nested JSON with json_normalize - great for deeply nested responses
df = pd.json_normalize(
    data['user']['repositories'], sep='_'  # Use underscore to separate nested keys
)
print("Simple flatten:")
print(df)

# Or access nested data with record_path and meta
# record_path = which nested list to expand as rows
# meta = which parent fields to carry along as columns
df = pd.json_normalize(
    data,
    record_path=['user', 'repositories'],
    meta=[['user', 'name'], ['user', 'address', 'city']],
    meta_prefix='user_',
)
print("\nWith metadata:")
print(df)

Simple flatten:
    name  stars
0  repo1     10
1  repo2     25

With metadata:
    name  stars user_user.name user_user.address.city
0  repo1     10          Ahmed                  Cairo
1  repo2     25          Ahmed                  Cairo


### <span style="color:blue">XML</span>

Some older APIs (e.g., government data, RSS feeds) return XML instead of JSON.

In [11]:
import xml.etree.ElementTree as ET

# Sample XML response from an API
xml_string = """
<library>
  <book id="1">
    <title>Python Basics</title>
    <author>John Doe</author>
    <year>2023</year>
  </book>
  <book id="2">
    <title>Data Science</title>
    <author>Jane Smith</author>
    <year>2024</year>
  </book>
</library>
"""

# Parse the XML string into a tree structure
root = ET.fromstring(xml_string)

# Navigate the tree and extract data into a list of dicts
books = []
for book in root.findall(
    './/book'
):  # './/book' finds all <book> tags anywhere in the tree
    books.append(
        {
            'id': book.get('id'),  # Get XML attribute
            'title': book.find('title').text,  # Get text content of child tag
            'author': book.find('author').text,
            'year': int(book.find('year').text),
        }
    )

# Convert to DataFrame for easy analysis
df = pd.DataFrame(books)
print(df)

  id          title      author  year
0  1  Python Basics    John Doe  2023
1  2   Data Science  Jane Smith  2024


---
## <span style="color:blue">2.9 Graded Exercise 2: GitHub Repository Analysis</span>

**Scenario**: Analyze GitHub repositories to understand popular technologies.

---

### 📋 Task 1: Repository Information *(15 points)*

**1.1** *(5 points)* Fetch information for these repositories:
- `tensorflow/tensorflow`
- `pytorch/pytorch`
- `scikit-learn/scikit-learn`

Create a **DataFrame** with columns: `name`, `stars`, `forks`, `language`, `open_issues`, `created_date`

Save as: **`task1_github.csv`**

In [16]:
import requests
import pandas as pd
import matplotlib.pyplot as plt


def task1_fetch_repos():
    """
    Fetch repository information for major ML frameworks.
    Returns a DataFrame with key metrics.
    """
    repos = ['tensorflow/tensorflow', 'pytorch/pytorch', 'scikit-learn/scikit-learn']
    rows = []

    for repo in repos:
        url = f'https://api.github.com/repos/{repo}'
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                data = response.json()
                rows.append(
                    {
                        'name': data['full_name'],
                        'stars': data['stargazers_count'],
                        'forks': data['forks_count'],
                        'language': data['language'],
                        'open_issues': data['open_issues_count'],
                        'created_date': data['created_at'],
                    }
                )

            else:
                print(f'failed to fetch {repo}, status code : {response.status_code}')
        except requests.RequestException as exc:
            print(f'error with fetching {repo}: {exc}')

    df = pd.DataFrame(rows)
    if not df.empty:
        df['created_date'] = pd.to_datetime(df['created_date'], utc=True).dt.date
    return df


df = task1_fetch_repos()
df.to_csv('task1_github.csv', index=False)

metrics_df = df.copy()
metrics_df['created_date'] = pd.to_datetime(metrics_df['created_date'], utc=True)
now_utc = pd.Timestamp.utcnow()


metrics_df['age_days'] = (now_utc - metrics_df['created_date']).dt.days.clip(lower=1)
metrics_df['stars_per_day'] = (metrics_df['stars'] / metrics_df['age_days'])
metrics_df['issues_per_star_ratio'] = (
    metrics_df['open_issues'] / metrics_df['stars'].replace(0, pd.NA)).fillna(0).round(4)


task1_metrics = metrics_df[
    ['name', 'age_days', 'stars_per_day', 'issues_per_star_ratio']
]
task1_metrics.to_csv('task1_metrics.csv', index=False)

plot_df = metrics_df[['name', 'stars', 'forks', 'stars_per_day']].copy()
plot_df = plot_df.sort_values('stars', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(plot_df['name'], plot_df['stars'])
axes[0].set_title('Stars')
axes[0].tick_params(axis='x', rotation=25)


axes[1].bar(plot_df['name'], plot_df['forks'])
axes[1].set_title('Forks')
axes[1].tick_params(axis='x', rotation=25)


axes[2].bar(plot_df['name'], plot_df['stars_per_day'])
axes[2].set_title('Stars Per Day')
axes[2].tick_params(axis='x', rotation=25)

fig.tight_layout()
fig.savefig('task1_comparison.png')
plt.close(fig)

print(df)
print(task1_metrics)

                        name   stars  forks language  open_issues created_date
0      tensorflow/tensorflow  194009  75218      C++         3616   2015-11-07
1            pytorch/pytorch   97991  27087   Python        18088   2016-08-13
2  scikit-learn/scikit-learn   65306  26749   Python         2133   2010-08-17
                        name  age_days  stars_per_day  issues_per_star_ratio
0      tensorflow/tensorflow      3772      51.433987                 0.0186
1            pytorch/pytorch      3492      28.061569                 0.1846
2  scikit-learn/scikit-learn      5680      11.497535                 0.0327


**1.2** *(5 points)* For each repo, calculate:
- **Age in days** (from `created_date` to now)
- **Stars per day**
- **Issues per star ratio**

Save as: **`task1_metrics.csv`**

**1.3** *(5 points)* Create a **visualization** comparing the three repositories:
- Save as: **`task1_comparison.png`**
- Use `matplotlib` or `seaborn`
- Compare **at least 3 metrics**

---

### 📋 Task 2: User Repository Analysis *(20 points)*

**2.1** *(10 points)* Choose any GitHub user and fetch **ALL** their repositories (handle pagination)

Requirements:
- Implement **pagination** properly
- Add **rate limiting** (wait 1 second between requests)
- Handle errors gracefully
- Log progress

Save as: **`task2_all_repos.csv`**

In [17]:
import time
import requests
import pandas as pd
import logging

logger = logging.getLogger('task2_github')
if not logger.handlers:
    logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler('api_requests.log', encoding='utf-8')
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(file_handler)


def fetch_user_repos_paginated(username):
    """
    Fetch all repositories for a user with pagination.

    Args:
        username: GitHub username

    Returns:
        list: All repositories
    """
    all_repos = []
    page = 1
    per_page = 100

    while True:
        url = f'https://api.github.com/users/{username}/repos'
        params = {'page': page, 'per_page': per_page, 'type': 'owner'}

        try:
            logger.info(f'fetching {username} repos - page {page}')
            print(f'fetching page {page}...')
            response = requests.get(url, params=params, timeout=10)

            if response.status_code == 404:
                logger.error(f'user not found: {username}')
                print(f'user not found: {username}')
                break

            if response.status_code != 200:
                logger.error(f'HTTP {response.status_code} on page {page}: {response.text[:200]}')
                print(f'Error HTTP {response.status_code} on page {page}')
                break

            data = response.json()
            if not data:
                logger.info(f'No more repositories after page {page - 1}')
                break

            all_repos.extend(data)
            logger.info(f'Page {page} fetched: {len(data)} repos (total: {len(all_repos)})')
            page += 1
            time.sleep(1)

        except requests.RequestException as exc:
            logger.exception(f'request failed on page {page}: {exc}')
            print(f'request failed on page {page}: {exc}')
            break

    return all_repos


def analyze_user_repos(repos):
    if not repos:
        return 'No repositories available for analysis.'

    df = pd.DataFrame(repos)

    language_series = df['language'].dropna()
    most_used_language = language_series.mode().iloc[0] if not language_series.empty else 'N/A'
    avg_stars = float(df['stargazers_count'].fillna(0).mean())
    total_forks = int(df['forks_count'].fillna(0).sum())

    df['updated_at'] = pd.to_datetime(df['updated_at'], errors='coerce', utc=True)
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', utc=True)

    newest_idx = df['updated_at'].idxmax()
    oldest_idx = df['created_at'].idxmin()

    most_recent_repo = df.loc[newest_idx, 'full_name'] if pd.notna(newest_idx) else 'N/A'
    oldest_repo = df.loc[oldest_idx, 'full_name'] if pd.notna(oldest_idx) else 'N/A'

    report = (
        f"most used programming language: {most_used_language}\n"
        f"average stars per repository: {avg_stars:.2f}\n"
        f"total forks of all repos: {total_forks}\n"
        f"most recently updated repo: {most_recent_repo}\n"
        f"oldest repo: {oldest_repo}\n"
    )
    return report


username = 'torvalds'
repos = fetch_user_repos_paginated(username)

repos_df = pd.DataFrame(
    [
        {
            'name': r.get('name'),
            'full_name': r.get('full_name'),
            'language': r.get('language'),
            'stargazers_count': r.get('stargazers_count', 0),
            'forks_count': r.get('forks_count', 0),
            'open_issues_count': r.get('open_issues_count', 0),
            'created_at': r.get('created_at'),
            'updated_at': r.get('updated_at'),
            'html_url': r.get('html_url'),
        }
        for r in repos
    ]
)
repos_df.to_csv('task2_all_repos.csv', index=False)

analysis_report = analyze_user_repos(repos)
with open('task2_analysis.txt', 'w', encoding='utf-8') as f:
    f.write(analysis_report)

print(f'fetched {len(repos_df)} repos for {username}')
print(analysis_report)

2026-03-06 13:09:38,617 - task2_github - INFO - fetching torvalds repos - page 1


fetching page 1...


2026-03-06 13:09:39,143 - task2_github - INFO - Page 1 fetched: 11 repos (total: 11)
2026-03-06 13:09:40,145 - task2_github - INFO - fetching torvalds repos - page 2


fetching page 2...


2026-03-06 13:09:40,505 - task2_github - INFO - No more repositories after page 1


fetched 11 repos for torvalds
most used programming language: C
average stars per repository: 21148.09
total forks of all repos: 61826
most recently updated repo: torvalds/linux
oldest repo: torvalds/linux



**2.2** *(10 points)* Analyze the repositories:
- Most used **programming language**
- **Average stars** per repository
- **Total forks** across all repos
- **Most recently updated** repo
- **Oldest** repo

Create a summary report saved as: **`task2_analysis.txt`**

---

### 📋 Task 3: Advanced API Client *(15 points)*

**3.1** *(15 points)* Build a complete `GitHubAnalyzer` class

Requirements:
- **Inherit from** or include rate limiting
- Implement **retry logic** with exponential backoff
- Add **logging**
- Include these methods:
  - `search_repos(query, language, min_stars)` — Search repositories
  - `get_trending(language, since)` — Get trending repos
  - `compare_repos(repo_list)` — Compare multiple repos
  - `export_to_excel(df, filename)` — Export with formatting

In [18]:
import os
import time
import logging
from datetime import datetime, timedelta, timezone

import requests
import pandas as pd
from openpyxl.styles import Font


class SimpleRateLimiter:

    def __init__(self, min_interval_seconds=1.0):
        self.min_interval_seconds = float(min_interval_seconds)
        self.last_request_time = 0.0

    def wait(self):
        now = time.time()
        elapsed = now - self.last_request_time
        if elapsed < self.min_interval_seconds:
            time.sleep(self.min_interval_seconds - elapsed)
        self.last_request_time = time.time()


class GitHubAnalyzer:

    def __init__(self, token=None, max_retries=4):
        self.base_url = 'https://api.github.com'
        self.max_retries = max_retries
        self.session = requests.Session()
        self.rate_limiter = SimpleRateLimiter(min_interval_seconds=1.0)

        self.logger = logging.getLogger(self.__class__.__name__)
        if not self.logger.handlers:
            self.logger.setLevel(logging.INFO)
            file_handler = logging.FileHandler('api_requests.log', encoding='utf-8')
            file_handler.setFormatter(
                logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
            )
            self.logger.addHandler(file_handler)

        token = token or os.getenv('GITHUB_TOKEN')
        if token:
            self.session.headers.update({'Authorization': f'Bearer {token}'})

        self.session.headers.update(
            {
                'Accept': 'application/vnd.github+json',
                'User-Agent': 'Lab03-GitHubAnalyzer',
            }
        )

    def _request(self, endpoint, params=None):
        url = f"{self.base_url}/{endpoint.lstrip('/')}"

        for attempt in range(self.max_retries):
            self.rate_limiter.wait()
            try:
                response = self.session.get(url, params=params, timeout=30)
                status = response.status_code

                if status == 200:
                    self.logger.info(f'GET {url} -> 200')
                    return response.json()

                if status == 429:
                    retry_after = int(response.headers.get('Retry-After', 2**attempt))
                    self.logger.warning(
                        f'limited reached on {url}; sleeping {retry_after}s (attempt {attempt + 1})'
                    )
                    time.sleep(retry_after)
                    continue

                if status >= 500:
                    backoff = 2**attempt
                    self.logger.warning(
                        f'server error {status} on {url}; backoff {backoff}s (attempt {attempt + 1})'
                    )
                    time.sleep(backoff)
                    continue

                self.logger.error(f'Client error {status} on {url}: {response.text[:200]}')
                return None

            except requests.RequestException as exc:
                backoff = 2**attempt
                self.logger.warning(
                    f'request exception on {url}: {exc}; backoff {backoff}s (attempt {attempt + 1})'
                )
                time.sleep(backoff)

        self.logger.error(f'failed after {self.max_retries} attempts: {url}')
        return None

    def search_repos(self, query, language=None, min_stars=0):

        q = query.strip()
        if language:
            q += f' language:{language}'
        if min_stars > 0:
            q += f' stars:>={int(min_stars)}'

        params = {'q': q, 'sort': 'stars', 'order': 'desc', 'per_page': 30}
        data = self._request('/search/repositories', params=params)

        if not data or 'items' not in data:
            return pd.DataFrame()

        rows = []
        for item in data['items']:
            rows.append(
                {
                    'name': item.get('full_name'),
                    'description': item.get('description'),
                    'language': item.get('language'),
                    'stars': item.get('stargazers_count', 0),
                    'forks': item.get('forks_count', 0),
                    'open_issues': item.get('open_issues_count', 0),
                    'updated_at': item.get('updated_at'),
                    'html_url': item.get('html_url'),
                }
            )

        return pd.DataFrame(rows)

    def get_trending(self, language=None, since='weekly'):

        if isinstance(since, int):
            days = max(1, since)
        else:
            since_map = {'daily': 1, 'weekly': 7, 'monthly': 30}
            days = since_map.get(str(since).lower(), 7)

        created_after = (datetime.now(timezone.utc) - timedelta(days=days)).strftime('%Y-%m-%d')
        q = f'created:>={created_after}'
        if language:
            q += f' language:{language}'

        params = {'q': q, 'sort': 'stars', 'order': 'desc', 'per_page': 30}
        data = self._request('/search/repositories', params=params)

        if not data or 'items' not in data:
            return pd.DataFrame()

        rows = []
        for item in data['items']:
            rows.append(
                {
                    'name': item.get('full_name'),
                    'language': item.get('language'),
                    'stars': item.get('stargazers_count', 0),
                    'forks': item.get('forks_count', 0),
                    'created_at': item.get('created_at'),
                    'html_url': item.get('html_url'),
                }
            )

        return pd.DataFrame(rows)

    def compare_repos(self, repo_list):

        rows = []
        for repo in repo_list:
            data = self._request(f'/repos/{repo}')
            if not data:
                continue

            rows.append(
                {
                    'name': data.get('full_name', repo),
                    'language': data.get('language'),
                    'stars': data.get('stargazers_count', 0),
                    'forks': data.get('forks_count', 0),
                    'open_issues': data.get('open_issues_count', 0),
                    'watchers': data.get('subscribers_count', 0),
                    'created_at': data.get('created_at'),
                    'updated_at': data.get('updated_at'),
                    'html_url': data.get('html_url'),
                }
            )

        return pd.DataFrame(rows)

    def export_to_excel(self, df, filename): # i used chatgpt for that
        """
        Export DataFrame to Excel with formatting.
        - Bold headers
        - Auto-adjust column widths
        - Add creation timestamp
        """
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, index=False, sheet_name='Results')
            ws = writer.book['Results']

            # Bold headers
            for cell in ws[1]:
                cell.font = Font(bold=True)

            # Auto-adjust column widths
            for col in ws.columns:
                max_len = 0
                col_letter = col[0].column_letter
                for cell in col:
                    value_len = len(str(cell.value)) if cell.value is not None else 0
                    max_len = max(max_len, value_len)
                ws.column_dimensions[col_letter].width = min(max_len + 2, 60)

            # Add creation timestamp after the table
            timestamp_row = len(df) + 3
            ws.cell(row=timestamp_row, column=1, value='Generated at (UTC)')
            ws.cell(
                row=timestamp_row,
                column=2,
                value=datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
            )


analyzer = GitHubAnalyzer()
search_results = analyzer.search_repos('data science', language='Python', min_stars=500)

repo_list = [
    'tensorflow/tensorflow',
    'pytorch/pytorch',
    'scikit-learn/scikit-learn',
    'numpy/numpy',
    'pandas-dev/pandas',
]
comparison_df = analyzer.compare_repos(repo_list)
analyzer.export_to_excel(comparison_df, 'task3_results.xlsx')

print('Search results sample:')
print(search_results.head())
print('\nComparison results:')
print(comparison_df)

2026-03-06 13:14:05,989 - GitHubAnalyzer - INFO - GET https://api.github.com/search/repositories -> 200
2026-03-06 13:14:06,262 - GitHubAnalyzer - INFO - GET https://api.github.com/repos/tensorflow/tensorflow -> 200
2026-03-06 13:14:07,298 - GitHubAnalyzer - INFO - GET https://api.github.com/repos/pytorch/pytorch -> 200
2026-03-06 13:14:08,275 - GitHubAnalyzer - INFO - GET https://api.github.com/repos/scikit-learn/scikit-learn -> 200
2026-03-06 13:14:09,251 - GitHubAnalyzer - INFO - GET https://api.github.com/repos/numpy/numpy -> 200
2026-03-06 13:14:10,251 - GitHubAnalyzer - INFO - GET https://api.github.com/repos/pandas-dev/pandas -> 200


Search results sample:
                                         name  \
0  donnemartin/data-science-ipython-notebooks   
1                             kedro-org/kedro   
2                            OpenMined/PySyft   
3     drivendataorg/cookiecutter-data-science   
4          joelgrus/data-science-from-scratch   

                                         description language  stars  forks  \
0  Data science Python notebooks: Deep learning (...   Python  28907   8033   
1  Kedro is a toolbox for production-ready data s...   Python  10779   1011   
2  Perform data science on data that remains in s...   Python   9856   2007   
3  A logical, reasonably standardized, but flexib...   Python   9706   2627   
4            code for Data Science From Scratch book   Python   9517   4724   

   open_issues            updated_at  \
0           42  2026-03-06T08:43:21Z   
1          200  2026-03-06T10:35:19Z   
2           66  2026-03-04T12:19:18Z   
3           29  2026-03-06T00:32:51Z   
4      

---
## 📦 Submission Requirements

### Files to submit:

| # | File | Description |
|---|---|---|
| 1 | `github_analysis.py` | All your code |
| 2 | `task1_github.csv` | Repo info table |
| 3 | `task1_metrics.csv` | Calculated metrics |
| 4 | `task1_comparison.png` | Visualization |
| 5 | `task2_all_repos.csv` | All user repos |
| 6 | `task2_analysis.txt` | Summary report |
| 7 | `task3_results.xlsx` | Excel export |
| 8 | `api_requests.log` | Your log file |
| 9 | `README.md` | Brief report of findings |

---

### 📊 Grading Rubric:

| Category | Weight |
|---|---|
| **Correct functionality** | 60% |
| **Code quality** (comments, error handling, logging) | 20% |
| **Output formatting** | 10% |
| **Analysis insights** | 10% |